Installs and Imports

In [3]:
!pip install wikipedia feedparser newsapi-python ntscraper snscrape atproto==0.0.65 tqdm langchain-text-splitters

In [4]:
import pandas as pd
import os
import wikipedia
import requests
import time
import random
import feedparser
import urllib.parse
import logging
import plotly.express as px
import importlib
import data_cleaning_1
import warnings

from datetime import datetime
from urllib.parse import quote
from newsapi import NewsApiClient
from datetime import datetime, timedelta
from atproto import Client
from atproto_client.models.app.bsky.feed.search_posts import Params as SearchParams
from tqdm import tqdm
from data_cleaning_1 import setup_cleaning_pipeline, clean_topic_column, EmojiRemover
from bs4 import GuessedAtParserWarning
from langchain_text_splitters import RecursiveCharacterTextSplitter

warnings.filterwarnings("ignore", category=GuessedAtParserWarning)

C:\Users\rucha\anaconda3\envs\py3115_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [5]:
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("urllib3").setLevel(logging.WARNING)
logging.getLogger("atproto").setLevel(logging.WARNING)

# Global Configuration & API keys

In [7]:
NEWS_API_KEY = "d0e5104607c542909e6b4710abaa74a1"
BSKY_HANDLE = 'ruchab11.bsky.social'
BSKY_PASSWORD = 'RB#20021113'
TRENDS_CSV = 'trends.csv'

# Load Trend Queries (The Seed)

In [9]:
# Load queries from CSV and return Top 10 and Top 100 lists
def load_trends(file_path):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Missing {file_path}. Please place it in {os.getcwd()}")

    df = pd.read_csv(file_path)
    all_queries = df['query'].tolist() if 'query' in df.columns else df.iloc[:, 0].tolist()

    return all_queries[:10], all_queries[:100]

In [10]:
# Initialize the query lists
top_10_queries, top_100_queries = load_trends(TRENDS_CSV)

print(f"Setup Complete")
print(f"Targeting {len(top_10_queries)} trends for Deep Context (Tier 1).")
print(f"Targeting {len(top_100_queries)} trends for Momentum (Tier 2).")

Setup Complete
Targeting 10 trends for Deep Context (Tier 1).
Targeting 100 trends for Momentum (Tier 2).


In [11]:
for i, query in enumerate(top_10_queries, 1):
        print(f"{i}. {query}")

1. 2026 winter olympics women's snowboarding halfpipe
2. mavericks vs lakers
3. atlético madrid - barcelona
4. cruz azul - vancouver football club
5. bucks vs thunder
6. cardi b tour
7. spider noir
8. latvia
9. brentford vs arsenal
10. alfonso ribeiro


# Data Collector Classes

## Wikipedia

In [14]:
class WikiCollector:
    def __init__(self):
        wikipedia.set_lang("en")

    def get_context(self, queries):
        """Tier 1: Factual Background"""
        results = []
        for topic in tqdm(queries, desc="Wikipedia Search"):
            try:
                search_results = wikipedia.search(topic)
                if not search_results:
                    continue

                best_match = search_results[0]
                summary = wikipedia.summary(best_match, sentences=3, auto_suggest=False)
                url = wikipedia.page(best_match, auto_suggest=False).url

                results.append({
                    'topic': topic,
                    'source': 'Wikipedia',
                    'title': best_match,
                    'text': summary,
                    'url': url
                })
            except Exception:
                continue
        return pd.DataFrame(results)

## News

In [16]:
class NewsCollector:
    def __init__(self, api_key):
        self.client = NewsApiClient(api_key=api_key)

    def get_context(self, queries, days=7):
        """Tier 1: Current Events"""
        results = []
        from_date = (datetime.now() - timedelta(days=days)).strftime('%Y-%m-%d')

        for topic in tqdm(queries, desc="NewsAPI Collection"):
            try:
                response = self.client.get_everything(
                    q=topic, from_param=from_date, language='en',
                    sort_by='publishedAt', page_size=5
                )
                articles = response.get('articles', [])
                for art in articles:
                    results.append({
                        'topic': topic,
                        'source': 'NewsAPI',
                        'title': art['title'],
                        'text': art.get('description', ''),
                        'url': art['url'],
                        'timestamp': pd.to_datetime(art['publishedAt']).tz_localize(None)
                    })
            except Exception:
                continue
        return pd.DataFrame(results)

## Reddit

In [18]:
class RedditCollector:
    def __init__(self):
        self.headers = {'User-Agent': 'VeloContextCollector/1.0 (Northeastern University Project)'}

    def get_data(self, queries, mode='full'):
        """
        mode='full': Tier 1 (Text + Metadata)
        mode='metadata': Tier 2 (Engagement scores only)
        """
        results = []
        limit = 5 if mode == 'full' else 1
        desc_text = "Reddit Full" if mode == 'full' else "Reddit Momentum"

        for topic in tqdm(queries, desc=desc_text):
            try:
                url = f"https://www.reddit.com/search.json?q={topic}&limit={limit}&sort=relevancy&t=day"
                resp = requests.get(url, headers=self.headers, timeout=10)
                if resp.status_code == 200:
                    posts = resp.json().get('data', {}).get('children', [])
                    for p in posts:
                        data = p['data']
                        entry = {
                            'topic': topic,
                            'source': 'Reddit',
                            'engagement': data.get('score', 0) + data.get('num_comments', 0),
                            'timestamp': datetime.fromtimestamp(data.get('created_utc', 0))
                        }
                        if mode == 'full':
                            entry.update({'title': data.get('title', ''), 'text': data.get('selftext', '')})
                        results.append(entry)
                time.sleep(1.2)
            except Exception:
                continue
        return pd.DataFrame(results)

## Twitter

In [20]:
class TwitterCollector:
    def __init__(self, handle, password):
        self.client = Client()
        self.client.login(handle, password)

    def get_data(self, queries, mode='full'):
        """
        mode='full': Tier 1 (Text + Metadata)
        mode='metadata': Tier 2 (Engagement scores only)
        """
        results = []
        limit = 10 if mode == 'full' else 1
        desc_text = "Twitter Full" if mode == 'full' else "Twitter Momentum"

        for topic in tqdm(queries, desc=desc_text):
            try:
                params = SearchParams(q=topic, limit=limit)
                resp = self.client.app.bsky.feed.search_posts(params=params)
                for post in resp.posts:
                    engagement = (post.like_count or 0) + (post.repost_count or 0) + (post.reply_count or 0)
                    entry = {
                        'topic': topic,
                        'source': 'Twitter',
                        'engagement': engagement,
                        'timestamp': pd.to_datetime(post.record.created_at).tz_localize(None)
                    }
                    if mode == 'full':
                        entry.update({'title': f"@{post.author.handle}", 'text': post.record.text})
                    results.append(entry)
                time.sleep(1)
            except Exception:
                continue
        return pd.DataFrame(results)

## Check data integrity

In [22]:
def run_integrity_audit(context_df, queries):
    """
    Analyzes the Top 10 trends against the 4 specific Cases:
    Group 1: Social (Twitter + Reddit) -> Sentiment
    Group 2: Factual (Wiki + News) -> Ground Truth/RAG
    """
    audit_results = []

    for topic in queries:
        # Filter data for this specific trend
        subset = context_df[context_df['topic'] == topic]
        found_sources = subset['source'].unique()

        # Check Group 1: Social Pulse
        has_social = 'Twitter' in found_sources or 'Reddit' in found_sources

        # Check Group 2: Ground Truth
        has_facts = 'Wikipedia' in found_sources or 'NewsAPI' in found_sources

        # Determine Status Case
        if has_social and has_facts:
            status = "Full Data (Ready)"
        elif not has_social and not has_facts:
            status = "Case 2/4: No Data Found"
        elif not has_social:
            status = "Case 2: Missing Social (No Sentiment)"
        else:
            status = "Case 4: Missing Facts (No RAG)"

        audit_results.append({
            'Trend': topic,
            'Audit Status': status,
            'Sources': ", ".join(found_sources) if len(found_sources) > 0 else "None"
        })

    return pd.DataFrame(audit_results)

# Tier 1 Execution (Top 10)

### Initialize Collectors

In [25]:
wiki_bot = WikiCollector()
news_bot = NewsCollector(NEWS_API_KEY)
reddit_bot = RedditCollector()
twitter_bot = TwitterCollector(BSKY_HANDLE, BSKY_PASSWORD)

### Collect from all sources

In [27]:
# Using mode='full' for social sources to get text
wiki_data = wiki_bot.get_context(top_10_queries)
news_data = news_bot.get_context(top_10_queries)
reddit_data = reddit_bot.get_data(top_10_queries, mode='full')
twitter_data = twitter_bot.get_data(top_10_queries, mode='full')

Twitter Full: 100%|██████████| 10/10 [00:12<00:00,  1.25s/it]


### Combine into a Master Contect DataFrame

In [29]:
tier1_dfs = [wiki_data, news_data, reddit_data, twitter_data]
master_context_df = pd.concat([df for df in tier1_dfs if not df.empty], ignore_index=True)

### Final cleanup and preview

In [31]:
if not master_context_df.empty:
    # Ensure all timestamps are uniform (removing timezone info for CSV compatibility)
    if 'timestamp' in master_context_df.columns:
        master_context_df['timestamp'] = pd.to_datetime(master_context_df['timestamp']).dt.tz_localize(None)

    # Save the result
    master_context_df.to_csv('tier1_context_master.csv', index=False)

    print("\n" + "="*30)
    print(f"TIER 1 COMPLETE")
    print(f"Total Records Collected: {len(master_context_df)}")
    print(f"Sources Represented: {master_context_df['source'].unique()}")
    print("="*30)
else:
    print("Warning: No data was collected for Tier 1.")


TIER 1 COMPLETE
Total Records Collected: 191
Sources Represented: ['Wikipedia' 'NewsAPI' 'Reddit' 'Twitter']


### Check data integrity

In [33]:
audit_report = run_integrity_audit(master_context_df, top_10_queries)

print("\n" + "="*60)
print("FINAL PROJECT DATA AUDIT: TIER 1 (TOP 10)")
print("="*60)
print(audit_report[['Trend', 'Audit Status']].to_string(index=False))
print("="*60)

# Final Success Confirmation
successful_trends = audit_report[audit_report['Audit Status'] == "Full Data (Ready)"].shape[0]
print(f"Project Status: {successful_trends}/10 Trends have 100% Data coverage.")


FINAL PROJECT DATA AUDIT: TIER 1 (TOP 10)
                                             Trend      Audit Status
2026 winter olympics women's snowboarding halfpipe Full Data (Ready)
                               mavericks vs lakers Full Data (Ready)
                       atlético madrid - barcelona Full Data (Ready)
               cruz azul - vancouver football club Full Data (Ready)
                                  bucks vs thunder Full Data (Ready)
                                      cardi b tour Full Data (Ready)
                                       spider noir Full Data (Ready)
                                            latvia Full Data (Ready)
                              brentford vs arsenal Full Data (Ready)
                                   alfonso ribeiro Full Data (Ready)
Project Status: 10/10 Trends have 100% Data coverage.


# Tier 2 Execution (Top 100)

### Collect lightweight metadata

In [36]:
# Only using Reddit and Twitter for momentum because they provide engagement scores
reddit_momentum = reddit_bot.get_data(top_100_queries, mode='metadata')
twitter_momentum = twitter_bot.get_data(top_100_queries, mode='metadata')

Twitter Momentum: 100%|██████████| 100/100 [02:00<00:00,  1.21s/it]


### Combine into a Momentum DataFrame

In [38]:
momentum_dfs = [reddit_momentum, twitter_momentum]
master_momentum_df = pd.concat([df for df in momentum_dfs if not df.empty], ignore_index=True)

###
Final cleanup and Export

In [40]:
if not master_momentum_df.empty:
    # Standardizing timestamps for Power BI
    if 'timestamp' in master_momentum_df.columns:
        # First, ensure it's a datetime object and strip timezone
        master_momentum_df['timestamp'] = pd.to_datetime(master_momentum_df['timestamp']).dt.tz_localize(None)

        # SECOND: Convert to the exact string format Power BI loves (YYYY-MM-DD HH:MM:SS)
        # This prevents Power BI from misinterpreting dates in different regions
        master_momentum_df['timestamp'] = master_momentum_df['timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S')

    # Save the result
    master_momentum_df.to_csv('tier2_momentum_master.csv', index=False)

    print("\n" + "="*30)
    print(f"TIER 2 COMPLETE")
    print(f"Total Momentum Signals: {len(master_momentum_df)}")
    print(f"File Saved: 'tier2_momentum_master.csv'")
    print("="*30)
else:
    print("Warning: No momentum data was collected.")


TIER 2 COMPLETE
Total Momentum Signals: 178
File Saved: 'tier2_momentum_master.csv'


### Visualize

In [42]:
def visualize_momentum(df):
    """
    Creates an interactive scatter plot showing
    Trend Engagement vs. Time for the Top 100.
    """
    # 1. Prepare the data (grouping by topic to get total engagement)
    viz_df = df.groupby('topic').agg({
        'engagement': 'sum',
        'timestamp': 'max'
    }).reset_index()

    # 2. Create the Plotly Scatter Plot
    fig = px.scatter(
        viz_df,
        x="topic",
        y="engagement",
        size="engagement", # Bigger bubble = more buzz
        color="engagement", # Color changes with intensity
        hover_name="topic",
        title="Tier 2: Trend Momentum Map (Top 100)",
        labels={'engagement': 'Social Buzz (Likes/Reposts)', 'topic': 'Trend Name'},
        template="plotly_dark"
    )

    # 3. Clean up the X-axis (hide labels if they overlap too much)
    fig.update_layout(xaxis_showticklabels=False)

    fig.show()

In [43]:
visualize_momentum(master_momentum_df)

# Data Cleaning

## Noise Removal and Text Normalization

In [46]:
importlib.reload(data_cleaning_1)

pipeline = setup_cleaning_pipeline()

In [47]:
master_context_df = pipeline.clean_dataframe(master_context_df, text_col='text', source_col='source')
master_context_df = clean_topic_column(master_context_df, pipeline)

# Drop empties
master_context_df_cleaned = master_context_df[
    (master_context_df['cleaned_text'].notna()) &
    (master_context_df['cleaned_text'].str.len() >= 5)
].copy()

# master_context_df_cleaned.to_csv('tier1_cleaned_data_final.csv', index=False)
# print("Saved complete dataset to 'tier1_cleaned_final.csv'")

In [48]:
emoji_stripper = EmojiRemover()

if 'title' in master_context_df_cleaned.columns:
    master_context_df_cleaned['title'] = master_context_df_cleaned['title'].apply(lambda x: emoji_stripper.clean(str(x)) if pd.notna(x) else x)

## Structural Standardization

### Ensure 'engagement' is numeric

In [51]:
if 'engagement' not in master_context_df_cleaned.columns:
    master_context_df_cleaned['engagement'] = 0
else:
    # Convert to numeric, turn errors to NaN, then fill NaN with 0
    master_context_df_cleaned['engagement'] = pd.to_numeric(master_context_df_cleaned['engagement'], errors='coerce').fillna(0)

### Advanced deduplication

In [53]:
master_context_df_cleaned = master_context_df_cleaned.drop_duplicates(subset=['cleaned_text'])

In [54]:
if 'title' in master_context_df_cleaned.columns:
    master_context_df_cleaned = master_context_df_cleaned.drop_duplicates(subset=['title', 'source'])

### Content density filter

In [56]:
# Remove rows that are too short (<20 char) to be useful
master_context_df_rag = master_context_df_cleaned[master_context_df_cleaned['cleaned_text'].str.len() > 20].copy()

### Metadata Enrichment (The "RAG Index")

In [58]:
def enrich_text_for_rag(row):
    """Bakes metadata into the text string for the Embedding model."""
    source = row['source']

    try:
        if pd.notna(row.get('timestamp')):
            date_val = pd.to_datetime(row['timestamp']).strftime('%Y-%m-%d')
        else:
            # Better label for static sources like Wikipedia
            date_val = "Static Context" if source == 'Wikipedia' else "Recent"
    except:
        date_val = "Recent"

    # Categorize Buzz for Tier 1 weighting (Crucial for your 24h Prediction)
    engagement = row.get('engagement', 0)
    if engagement > 500: buzz_level = "Very High"
    elif engagement > 100: buzz_level = "High"
    elif engagement > 20: buzz_level = "Moderate"
    else: buzz_level = "Low"

    # Create the Enriched String (Source + Date + Buzz)
    if source in ['Twitter', 'Reddit']:
        prefix = f"[Source: {source} | Date: {date_val} | Buzz: {buzz_level}] "
    else:
        prefix = f"[Source: {source} | Date: {date_val}] "

    return prefix + str(row['cleaned_text'])

In [59]:
# Apply the enrichment to create the final RAG input column
master_context_df_rag['rag_input_text'] = master_context_df_rag.apply(enrich_text_for_rag, axis=1)

## Final Save

In [61]:
master_context_df_rag.to_csv('tier1_rag_ready.csv', index=False)

print("\n" + "=" * 20)
print("PARTS 3 & 4 COMPLETE: DATA IS RAG-READY")
print(f"Final Record Count: {len(master_context_df_rag)}")
print(f"File Saved: 'tier1_rag_ready.csv'")
print("=" * 20)

# Quick Preview of the 'Baked' context
print("\nPreview of RAG Input Text:")
print(master_context_df_rag['rag_input_text'].iloc[0][:160] + "...")


PARTS 3 & 4 COMPLETE: DATA IS RAG-READY
Final Record Count: 145
File Saved: 'tier1_rag_ready.csv'

Preview of RAG Input Text:
[Source: Wikipedia | Date: Static Context] The women's halfpipe competition in snowboarding at the 2026 Winter Olympics was held on 11 February (qualification) ...


# Text Splitter

In [63]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=60,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

In [64]:
chunked_data = []

In [65]:
for _, row in master_context_df_rag.iterrows():
    full_text = row['rag_input_text']
    chunks = text_splitter.split_text(full_text)

    for i, chunk in enumerate(chunks):
        chunk_entry = row.to_dict() # Keep metadata (Source, Topic, Buzz)
        chunk_entry['chunk_text'] = chunk
        chunk_entry['chunk_id'] = f"{row.get('topic', 'unknown')}_{row.name}_{i}"
        chunked_data.append(chunk_entry)

In [66]:
df_chunked = pd.DataFrame(chunked_data)

In [67]:
df_chunked.to_csv('tier1_chunked_final.csv', index=False)

print(f"Chunking Complete!")
print(f"Original Rows: {len(master_context_df_rag)}")
print(f"Generated Chunks: {len(df_chunked)}")

Chunking Complete!
Original Rows: 145
Generated Chunks: 253


# Embedding Model

## EDA

In [70]:
import pandas as pd

df = pd.read_csv('tier1_chunked_final.csv')

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nData Types:\n", df.dtypes)
print("\nNull Counts:\n", df.isnull().sum())

Shape: (253, 12)

Columns: ['topic', 'source', 'title', 'text', 'url', 'timestamp', 'engagement', 'cleaned_text', 'topic_cleaned', 'rag_input_text', 'chunk_text', 'chunk_id']

Data Types:
 topic              object
source             object
title              object
text               object
url                object
timestamp          object
engagement        float64
cleaned_text       object
topic_cleaned      object
rag_input_text     object
chunk_text         object
chunk_id           object
dtype: object

Null Counts:
 topic               0
source              0
title               0
text                0
url               203
timestamp          11
engagement          0
cleaned_text        0
topic_cleaned       0
rag_input_text      0
chunk_text          0
chunk_id            0
dtype: int64


In [71]:
print("Chunks per Source:")
print(df['source'].value_counts())

print("\nChunks per Topic:")
print(df['topic'].value_counts())

Chunks per Source:
source
Reddit       134
Twitter       69
NewsAPI       39
Wikipedia     11
Name: count, dtype: int64

Chunks per Topic:
topic
brentford vs arsenal                                  54
atlético madrid - barcelona                           53
cardi b tour                                          26
latvia                                                23
cruz azul - vancouver football club                   21
2026 winter olympics women's snowboarding halfpipe    18
spider noir                                           17
alfonso ribeiro                                       17
bucks vs thunder                                      14
mavericks vs lakers                                   10
Name: count, dtype: int64


In [72]:
print("Sample Chunks by Source\n")
for source in df['source'].unique():
    sample = df[df['source'] == source]['chunk_text'].iloc[0]
    print(f"[{source}]")
    print(sample)
    print("-" * 60)

Sample Chunks by Source

[Wikipedia]
[Source: Wikipedia | Date: Static Context] The women's halfpipe competition in snowboarding at the 2026 Winter Olympics was held on 11 February (qualification) and 12 February (final), at the Livigno Snow Park in Valtellina. Choi Ga-on of South Korea won the event, the first gold medal for her. Chloe Kim of the United States won the silver medal and Mitsuki Ono of Japan won the bronze.
------------------------------------------------------------
[NewsAPI]
[Source: NewsAPI | Date: 2026-02-19] A storm dumping more than a half-foot of snow in the mountain town of Livigno on Thursday morning forced the postponement of Olympic men's ski halfpipe qualifying and men's aerials in the latest juggling of the snowboarding and freestyle schedules. The men's.
------------------------------------------------------------
[Reddit]
[Source: Reddit | Date: 2026-02-20 | Buzz: High] Welcome to the first game in the Men's Semifinals! Kivi, Lehky, and Team Finland go up 

In [73]:
# Boxscores have short "words" like "0", "1", "CAN", "USA" repeatedly
# A better signal is average word length - natural language averages 4-5 chars
df['avg_word_len'] = df['chunk_text'].apply(
    lambda x: sum(len(w) for w in x.split()) / max(len(x.split()), 1)
)

print("Chunks with avg word length < 3 (likely structured/tabular data):")
noisy = df[df['avg_word_len'] < 3]
print(f"Count: {len(noisy)}")
print("\nSamples:")
for _, row in noisy.head(3).iterrows():
    print(f"[{row['source']} | Topic: {row['topic']}]")
    print(row['chunk_text'][:150])
    print()

Chunks with avg word length < 3 (likely structured/tabular data):
Count: 0

Samples:


In [74]:
# 1. Duplicate chunk_text
dup_text = df.duplicated(subset=['chunk_text']).sum()
print(f"1. Duplicate chunk_text: {dup_text}")

# 2. chunk_id uniqueness
dup_ids = df.duplicated(subset=['chunk_id']).sum()
print(f"2. Duplicate chunk_ids: {dup_ids}")

# 3. Does chunk_text still contain the source prefix?
has_prefix = df['chunk_text'].str.startswith('[Source:').sum()
print(f"3. Chunks with [Source:] prefix: {has_prefix}/{len(df)}")

# 4. Very short chunks (under 50 chars)
short_chunks = df[df['chunk_text'].str.len() < 50]
print(f"4. Very short chunks (<50 chars): {len(short_chunks)}")
if len(short_chunks) > 0:
    print(short_chunks['chunk_text'].tolist())

# 5. Topic casing consistency
print(f"5. Unique topic values:")
for t in sorted(df['topic'].unique()):
    print(f"   '{t}'")

1. Duplicate chunk_text: 0
2. Duplicate chunk_ids: 0
3. Chunks with [Source:] prefix: 145/253
4. Very short chunks (<50 chars): 0
5. Unique topic values:
   '2026 winter olympics women's snowboarding halfpipe'
   'alfonso ribeiro'
   'atlético madrid - barcelona'
   'brentford vs arsenal'
   'bucks vs thunder'
   'cardi b tour'
   'cruz azul - vancouver football club'
   'latvia'
   'mavericks vs lakers'
   'spider noir'


In [75]:
# Separate prefixed vs non-prefixed chunks
no_prefix = df[~df['chunk_text'].str.startswith('[Source:')]

print(f"Non-prefixed chunks by source:")
print(no_prefix['source'].value_counts())

print(f"\nNon-prefixed chunks by topic:")
print(no_prefix['topic'].value_counts())

print(f"\nSample non-prefixed chunks:")
for _, row in no_prefix.head(3).iterrows():
    print(f"\n[{row['source']} | {row['topic']}]")
    print(row['chunk_text'][:200])
    print("-" * 50)

Non-prefixed chunks by source:
source
Reddit       107
Wikipedia      1
Name: count, dtype: int64

Non-prefixed chunks by topic:
topic
brentford vs arsenal                                  36
atlético madrid - barcelona                           34
cruz azul - vancouver football club                   13
latvia                                                 8
cardi b tour                                           6
2026 winter olympics women's snowboarding halfpipe     4
alfonso ribeiro                                        4
bucks vs thunder                                       3
Name: count, dtype: int64

Sample non-prefixed chunks:

[Wikipedia | atlético madrid - barcelona]
on the final weekend of play (the others in 1945–46 and 1950–51 each involved one of the clubs involved in 2014, with Sevilla the other club on both occasions).
--------------------------------------------------

[Reddit | 2026 winter olympics women's snowboarding halfpipe]
GOALS: (16:55) M. Rantanen (PPG), As

In [76]:
# The boxscore chunks all contain this very specific pattern
# of team codes + numbers + hockey terminology
boxscore_mask = (
    df['chunk_text'].str.contains('Period Time Team', na=False) |
    df['chunk_text'].str.contains('Too Many Players', na=False) |
    df['chunk_text'].str.contains('FO Wins PPG', na=False)
)

boxscore_chunks = df[boxscore_mask]
print(f"Boxscore chunks identified: {len(boxscore_chunks)}")
print(f"\nTopics affected: {boxscore_chunks['topic'].unique()}")
print(f"Sources affected: {boxscore_chunks['source'].unique()}")
print(f"\nChunk IDs to be dropped:")
print(boxscore_chunks['chunk_id'].tolist())

Boxscore chunks identified: 0

Topics affected: []
Sources affected: []

Chunk IDs to be dropped:
[]


In [77]:
remaining = df[
    (df['topic'] == "2026 winter olympics women's snowboarding halfpipe") & 
    (~boxscore_mask)
]
print(f"Remaining chunks for snowboarding topic: {len(remaining)}")
print(remaining['source'].value_counts())

Remaining chunks for snowboarding topic: 18
source
Twitter      9
Reddit       5
NewsAPI      3
Wikipedia    1
Name: count, dtype: int64


## Installing BGE-M3

In [79]:
import sys
!{sys.executable} -m pip install "transformers==4.41.2" "tokenizers==0.19.1" "huggingface-hub==0.24.7" "sentence-transformers==3.0.1" "accelerate==0.33.0" "chromadb"


In [80]:
import os
import sys
import torch
import numpy as np
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer

# Suppress symlinks warning on Windows
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

print(f"torch: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
print(f"Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")

# Load BGE-M3 (~570MB download on first run, cached after)
model = SentenceTransformer('BAAI/bge-m3')

print(f"Model loaded: BAAI/bge-m3")
print(f"Embedding dimensions: {model.get_sentence_embedding_dimension()}")


torch: 2.10.0+cpu
GPU available: False
Device: CPU


C:\Users\rucha\anaconda3\envs\py3115_env\Lib\site-packages\huggingface_hub\file_download.py:1150: FutureWarning:

`resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.



Model loaded: BAAI/bge-m3
Embedding dimensions: 1024


## Embedding and Semantic filtering

In [82]:
# Load data
df_clean = pd.read_csv('tier1_chunked_final.csv').reset_index(drop=True)
print(f"Total chunks loaded: {len(df_clean)}")

# Embed all chunks 
print("\nEmbedding all chunks...")
embeddings = model.encode(
    df_clean['chunk_text'].tolist(),
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True
)
print(f"Embeddings shape: {embeddings.shape}")

# Embed topic names for similarity comparison
topics = df_clean['topic'].unique().tolist()
topic_vec_map = dict(zip(topics, model.encode(topics, normalize_embeddings=True)))

# Compute cosine similarity between each chunk and its topic
similarity_scores = []
for i, (_, row) in enumerate(df_clean.iterrows()):
    score = float(np.dot(embeddings[i], topic_vec_map[row['topic']]))
    similarity_scores.append(score)

df_clean['similarity_score'] = similarity_scores

# Apply threshold filter
THRESHOLD = 0.30
kept_indices = df_clean[df_clean['similarity_score'] >= THRESHOLD].index.tolist()
df_filtered = df_clean.iloc[kept_indices].reset_index(drop=True)
embeddings_filtered = embeddings[kept_indices]

print(f"\nBefore filter: {len(df_clean)} chunks")
print(f"After filter:  {len(df_filtered)} chunks")
print(f"Removed:       {len(df_clean) - len(df_filtered)} mismatched chunks")
print(f"Embeddings in sync: {'Good' if len(df_filtered) == embeddings_filtered.shape[0] else 'Check Again'}")



Total chunks loaded: 253

Embedding all chunks...


Batches: 100%|██████████| 8/8 [04:46<00:00, 35.76s/it]


Embeddings shape: (253, 1024)

Before filter: 253 chunks
After filter:  229 chunks
Removed:       24 mismatched chunks
Embeddings in sync: Good


### test

In [139]:
print("Chunks per topic:")
print(df_filtered['topic'].value_counts())

print("\nChunks per source:")
print(df_filtered['source'].value_counts())

missing = set(df_clean['topic'].unique()) - set(df_filtered['topic'].unique())
print(f"\nTopics with zero coverage: {missing if missing else 'None '}")
print(f"Embeddings in sync: {'Good' if len(df_filtered) == embeddings_filtered.shape[0] else 'Check Again'}")



Chunks per topic:
topic
brentford vs arsenal                                  54
atlético madrid - barcelona                           49
cardi b tour                                          26
2026 winter olympics women's snowboarding halfpipe    18
latvia                                                18
spider noir                                           17
cruz azul - vancouver football club                   14
bucks vs thunder                                      14
mavericks vs lakers                                   10
alfonso ribeiro                                        9
Name: count, dtype: int64

Chunks per source:
source
Reddit       120
Twitter       67
NewsAPI       31
Wikipedia     11
Name: count, dtype: int64

Topics with zero coverage: None 
Embeddings in sync: Good


## Storing in ChromaDB

In [86]:
client = chromadb.PersistentClient(path="./chroma_store")

# Always start fresh to avoid stale data
try:
    client.delete_collection("trend_chunks")
except:
    pass
collection = client.get_or_create_collection(
    name="trend_chunks",
    metadata={"hnsw:space": "cosine"}
)

# Prepare metadata
ids, metadatas = [], []
for _, row in df_filtered.iterrows():
    ids.append(str(row['chunk_id']))
    metadatas.append({
        'topic':            str(row['topic']),
        'source':           str(row['source']),
        'engagement':       float(row['engagement']),
        'timestamp':        str(row['timestamp']) if pd.notna(row['timestamp']) else 'Static Context',
        'similarity_score': float(row['similarity_score'])
    })

# Store vectors + metadata + documents
collection.add(
    ids=ids,
    embeddings=embeddings_filtered.tolist(),
    documents=df_filtered['chunk_text'].tolist(),
    metadatas=metadatas
)

print(f"df_filtered rows:    {len(df_filtered)}")
print(f"embeddings_filtered: {embeddings_filtered.shape[0]}")
print(f"ChromaDB vectors:    {collection.count()}")
print(f"All in sync: {'Check' if len(df_filtered) == embeddings_filtered.shape[0] == collection.count() else 'Check Again'}")

df_filtered rows:    229
embeddings_filtered: 229
ChromaDB vectors:    229
All in sync: Check


In [87]:
print("Final Test")
for topic in df_filtered['topic'].unique():
    results = collection.query(
        query_embeddings=model.encode([topic], normalize_embeddings=True).tolist(),
        n_results=2,
        where={"topic": {"$eq": topic}}
    )
    sources = [m['source'] for m in results['metadatas'][0]]
    print(f"{topic}")
    print(f"   Sources retrieved: {sources}\n")

print("Wiki + News")
r = collection.query(
    query_embeddings=model.encode(["why is cardi b tour trending"], normalize_embeddings=True).tolist(),
    n_results=3,
    where={"source": {"$in": ["Wikipedia", "NewsAPI"]}}
)
for doc, meta in zip(r['documents'][0], r['metadatas'][0]):
    print(f"[{meta['source']}] {doc[:80]}...")

print("\nTwitter + Reddit")
r = collection.query(
    query_embeddings=model.encode(["cardi b tour sentiment"], normalize_embeddings=True).tolist(),
    n_results=3,
    where={"source": {"$in": ["Twitter", "Reddit"]}}
)
for doc, meta in zip(r['documents'][0], r['metadatas'][0]):
    print(f"[{meta['source']} | engagement: {meta['engagement']}] {doc[:80]}...")

print(f"\nChromaDB persisted at: ./chroma_store")
print(f"Collection name: trend_chunks")
print(f"Total vectors: {collection.count()}")
print(f"Dimensions: 1024 (BGE-M3)")

Final Test
2026 winter olympics women's snowboarding halfpipe
   Sources retrieved: ['Wikipedia', 'Twitter']

mavericks vs lakers
   Sources retrieved: ['Twitter', 'Twitter']

atlético madrid - barcelona
   Sources retrieved: ['Wikipedia', 'Twitter']

cruz azul - vancouver football club
   Sources retrieved: ['Wikipedia', 'Twitter']

bucks vs thunder
   Sources retrieved: ['Twitter', 'Twitter']

cardi b tour
   Sources retrieved: ['Twitter', 'Twitter']

spider noir
   Sources retrieved: ['Wikipedia', 'Twitter']

latvia
   Sources retrieved: ['Twitter', 'Wikipedia']

brentford vs arsenal
   Sources retrieved: ['Twitter', 'Reddit']

alfonso ribeiro
   Sources retrieved: ['Twitter', 'Twitter']

Wiki + News
[NewsAPI] [Source: NewsAPI | Date: 2026-02-19] We all know that Cardi B has a lot of hater...
[Wikipedia] [Source: Wikipedia | Date: Static Context] American rapper Cardi B has released ...
[NewsAPI] [Source: NewsAPI | Date: 2026-02-19] The rapper kicked off her Am I The Drama? t...

Tw

### For future access 

In [160]:
#import chromadb

#client = chromadb.PersistentClient(path="./chroma_store")

#collection = client.get_collection("trend_chunks")

#results = collection.query(
    #query_embeddings=model.encode(["why is spider noir trending"], normalize_embeddings=True).tolist(),
    #n_results=3,
    #where={"source": {"$in": ["Wikipedia", "NewsAPI"]}}
#)

## Final Validation

In [143]:
# TEST 1: Verify source filter works for ALL sources independently
print("TEST 1: Source Filter Integrity\n")

for source in ['Wikipedia', 'NewsAPI', 'Twitter', 'Reddit']:
    results = collection.query(
        query_embeddings=model.encode(
            ["trending topic"], normalize_embeddings=True
        ).tolist(),
        n_results=3,
        where={"source": {"$eq": source}}
    )
    returned_sources = [m['source'] for m in results['metadatas'][0]]
    all_correct = all(s == source for s in returned_sources)
    print(f"[{source}] Filter accurate: {'Perfect' if all_correct else 'Check again'}")
    print(f"  Returned sources: {returned_sources}")

# TEST 2: Verify engagement metadata stored correctly
print("\n TEST 2: Engagement Metadata Integrity \n")
results = collection.query(
    query_embeddings=model.encode(
        ["trending"], normalize_embeddings=True
    ).tolist(),
    n_results=5
)
for meta in results['metadatas'][0]:
    eng = meta['engagement']
    print(f"[{meta['source']} | {meta['topic']}] engagement: {eng} — {'perfect' if isinstance(eng, float) else 'check again'}")

# TEST 3: Verify similarity scores stored correctly
print("\n TEST 3: Similarity Score Metadata Integrity \n")
for meta in results['metadatas'][0]:
    score = meta['similarity_score']
    valid = isinstance(score, float) and 0.30 <= score <= 1.0
    print(f"[{meta['topic']}] similarity_score: {score:.4f} — {'perfect' if valid else 'check again'}")

TEST 1: Source Filter Integrity

[Wikipedia] Filter accurate: Perfect
  Returned sources: ['Wikipedia', 'Wikipedia', 'Wikipedia']
[NewsAPI] Filter accurate: Perfect
  Returned sources: ['NewsAPI', 'NewsAPI', 'NewsAPI']
[Twitter] Filter accurate: Perfect
  Returned sources: ['Twitter', 'Twitter', 'Twitter']
[Reddit] Filter accurate: Perfect
  Returned sources: ['Reddit', 'Reddit', 'Reddit']

 TEST 2: Engagement Metadata Integrity 

[Twitter | brentford vs arsenal] engagement: 13.0 — perfect
[Twitter | cardi b tour] engagement: 0.0 — perfect
[Reddit | cruz azul - vancouver football club] engagement: 391.0 — perfect
[Reddit | cruz azul - vancouver football club] engagement: 391.0 — perfect
[NewsAPI | 2026 winter olympics women's snowboarding halfpipe] engagement: 0.0 — perfect

 TEST 3: Similarity Score Metadata Integrity 

[brentford vs arsenal] similarity_score: 0.3440 — perfect
[cardi b tour] similarity_score: 0.5793 — perfect
[cruz azul - vancouver football club] similarity_score: 0.3

In [145]:
# TEST 4: Step 6 simulation for ALL 10 topics
print(" TEST 4: Step 6 (Wiki+News) for All Topics \n")
for topic in df_filtered['topic'].unique():
    results = collection.query(
        query_embeddings=model.encode(
            [topic], normalize_embeddings=True
        ).tolist(),
        n_results=3,
        where={"source": {"$in": ["Wikipedia", "NewsAPI"]}}
    )
    docs = results['documents'][0]
    sources = [m['source'] for m in results['metadatas'][0]]
    print(f"[{topic}]")
    print(f"  Sources: {sources}")
    print(f"  Coverage: {'Perfect' if len(docs) > 0 else 'check again NO FACTUAL DATA'}")

# TEST 5: Step 7 simulation for ALL 10 topics
print("\n TEST 5: Step 7 (Twitter+Reddit) for All Topics \n")
for topic in df_filtered['topic'].unique():
    results = collection.query(
        query_embeddings=model.encode(
            [topic], normalize_embeddings=True
        ).tolist(),
        n_results=3,
        where={"source": {"$in": ["Twitter", "Reddit"]}}
    )
    docs = results['documents'][0]
    sources = [m['source'] for m in results['metadatas'][0]]
    print(f"[{topic}]")
    print(f"  Sources: {sources}")
    print(f"  Coverage: {'Perfect' if len(docs) > 0 else 'Check Again NO SOCIAL DATA'}")

 TEST 4: Step 6 (Wiki+News) for All Topics 

[2026 winter olympics women's snowboarding halfpipe]
  Sources: ['Wikipedia', 'NewsAPI', 'NewsAPI']
  Coverage: Perfect
[mavericks vs lakers]
  Sources: ['Wikipedia', 'NewsAPI', 'NewsAPI']
  Coverage: Perfect
[atlético madrid - barcelona]
  Sources: ['Wikipedia', 'NewsAPI', 'Wikipedia']
  Coverage: Perfect
[cruz azul - vancouver football club]
  Sources: ['Wikipedia', 'NewsAPI', 'Wikipedia']
  Coverage: Perfect
[bucks vs thunder]
  Sources: ['NewsAPI', 'NewsAPI', 'NewsAPI']
  Coverage: Perfect
[cardi b tour]
  Sources: ['NewsAPI', 'Wikipedia', 'NewsAPI']
  Coverage: Perfect
[spider noir]
  Sources: ['Wikipedia', 'NewsAPI', 'NewsAPI']
  Coverage: Perfect
[latvia]
  Sources: ['Wikipedia', 'NewsAPI', 'NewsAPI']
  Coverage: Perfect
[brentford vs arsenal]
  Sources: ['Wikipedia', 'NewsAPI', 'NewsAPI']
  Coverage: Perfect
[alfonso ribeiro]
  Sources: ['Wikipedia', 'NewsAPI', 'NewsAPI']
  Coverage: Perfect

 TEST 5: Step 7 (Twitter+Reddit) for All 

### Checking only engagement values

In [148]:
# Check engagement values in df_filtered (what we actually stored)
print("Engagement in df_filtered:")
print(df_filtered.groupby('source')['engagement'].agg(['mean', 'max', 'sum']))

print(f"\nNon-zero engagement in df_filtered: {(df_filtered['engagement'] > 0).sum()}")
print(f"Zero engagement in df_filtered: {(df_filtered['engagement'] == 0).sum()}")

# Verify ChromaDB stored them correctly by fetching Reddit chunks
print("\nSample Reddit chunks with engagement from ChromaDB:")
results = collection.query(
    query_embeddings=model.encode(
        ["trending"], normalize_embeddings=True
    ).tolist(),
    n_results=5,
    where={"source": {"$eq": "Reddit"}}
)
for meta in results['metadatas'][0]:
    print(f"  [{meta['topic']}] engagement: {meta['engagement']}")

Engagement in df_filtered:
                mean    max     sum
source                             
NewsAPI     0.000000    0.0     0.0
Reddit     63.133333  391.0  7576.0
Twitter     4.552239  113.0   305.0
Wikipedia   0.000000    0.0     0.0

Non-zero engagement in df_filtered: 150
Zero engagement in df_filtered: 79

Sample Reddit chunks with engagement from ChromaDB:
  [cruz azul - vancouver football club] engagement: 391.0
  [cruz azul - vancouver football club] engagement: 391.0
  [cardi b tour] engagement: 3.0
  [cardi b tour] engagement: 60.0
  [mavericks vs lakers] engagement: 13.0


In [150]:
# TEST 6: Verify high engagement chunks are retrievable
print(" TEST 6: Top Engagement Chunks \n")
top_eng = df_filtered.nlargest(5, 'engagement')[['topic', 'source', 'engagement', 'chunk_text']]
for _, row in top_eng.iterrows():
    print(f"[{row['source']} | {row['topic']}]")
    print(f"  Engagement: {row['engagement']}")
    print(f"  Text: {row['chunk_text'][:80]}...")
    print()

# TEST 7: Cross topic isolation - make sure querying one topic 
# doesn't return chunks from a completely unrelated topic
print(" TEST 7: Topic Isolation \n")
pairs = [
    ("cardi b tour", "atlético madrid - barcelona"),
    ("spider noir", "bucks vs thunder"),
    ("latvia", "mavericks vs lakers")
]
for topic_a, topic_b in pairs:
    results = collection.query(
        query_embeddings=model.encode(
            [topic_a], normalize_embeddings=True
        ).tolist(),
        n_results=5,
        where={"topic": {"$eq": topic_b}}
    )
    docs = results['documents'][0]
    print(f"Query: '{topic_a}' filtered to '{topic_b}'")
    print(f"  Results returned: {len(docs)} — {'Perfect Isolated correctly' if len(docs) > 0 else 'Warning'}")
    topics_returned = [m['topic'] for m in results['metadatas'][0]]
    print(f"  Topics in results: {set(topics_returned)}\n")

 TEST 6: Top Engagement Chunks 

[Reddit | cruz azul - vancouver football club]
  Engagement: 391.0
  Text: [Source: Reddit | Date: 2026-02-20 | Buzz: High] NAH JUST FUCKING WITH YOU. I'VE...

[Reddit | cruz azul - vancouver football club]
  Engagement: 391.0
  Text: BUT HAD NOTHING BUT SHIT TO SLING AT FOR NOT COMPLETELY ALIGNING WITH THEIR PEEW...

[Reddit | cruz azul - vancouver football club]
  Engagement: 391.0
  Text: YOU, JUST STOP); OR WIEBE WHO HAS THE ON-SCREEN PERSONALITY OF COTTAGE CHEESE (A...

[Reddit | cruz azul - vancouver football club]
  Engagement: 391.0
  Text: TIME UNTIL THEY SHAKEDOWN THE STREET TO VEGAS EVERY OTHER AFTERTHOUGHT TEAM IN T...

[Reddit | cruz azul - vancouver football club]
  Engagement: 391.0
  Text: OUR WAY TO GOING FULL MINORITY REPORT WITH THE NEW FEDERAL DATABASE GATHERING TO...

 TEST 7: Topic Isolation 

Query: 'cardi b tour' filtered to 'atlético madrid - barcelona'
  Results returned: 5 — Perfect Isolated correctly
  Topics in results: {'at

Why It's Dangerous
The engagement score of 12,735 means if your teammate's RAG engine ranks retrieved chunks by engagement, this chunk would always rank first for the snowboarding topic. The LLM would then read this as the most important context and potentially generate a summary saying something like "the snowboarding trend is notable for concerns about data accuracy" — a complete hallucination caused by bad data.
The Real Fix
This is actually a chunking problem. The original Reddit post was long enough that when split, one chunk lost its snowboarding context entirely. The proper fix would be at the chunking stage — adding a post-split relevance check before saving to CSV.

In [153]:
# TEST 8: Timestamp metadata integrity
print(" TEST 8: Timestamp Integrity \n")
for source in ['Wikipedia', 'NewsAPI', 'Twitter', 'Reddit']:
    results = collection.query(
        query_embeddings=model.encode(
            ["trending"], normalize_embeddings=True
        ).tolist(),
        n_results=2,
        where={"source": {"$eq": source}}
    )
    for meta in results['metadatas'][0]:
        ts = meta['timestamp']
        valid = ts == 'Static Context' or ts.startswith('2026')
        print(f"[{source}] timestamp: {ts} — {'Perfect' if valid else 'Check Again'}")

# TEST 9: Full pipeline simulation - exactly how teammate will use it
print("\n TEST 9: Full RAG Pipeline Simulation \n")
query = "why is spider noir trending right now"

#Summary generation context
factual = collection.query(
    query_embeddings=model.encode([query], normalize_embeddings=True).tolist(),
    n_results=3,
    where={"source": {"$in": ["Wikipedia", "NewsAPI"]}}
)

#Sentiment context  
social = collection.query(
    query_embeddings=model.encode([query], normalize_embeddings=True).tolist(),
    n_results=3,
    where={"source": {"$in": ["Twitter", "Reddit"]}}
)

print(f"Query: '{query}'\n")
print("Factual context (for summary):")
for doc, meta in zip(factual['documents'][0], factual['metadatas'][0]):
    print(f"  [{meta['source']}] {doc[:100]}...")

print("\nSocial context (for sentiment):")
for doc, meta in zip(social['documents'][0], social['metadatas'][0]):
    print(f"  [{meta['source']} | engagement: {meta['engagement']}] {doc[:100]}...")

 TEST 8: Timestamp Integrity 

[Wikipedia] timestamp: Static Context — Perfect
[Wikipedia] timestamp: Static Context — Perfect
[NewsAPI] timestamp: 2026-02-18 22:00:00.000000000 — Perfect
[NewsAPI] timestamp: 2026-02-18 19:44:58.000000000 — Perfect
[Twitter] timestamp: 2026-02-20 11:23:58.560000000 — Perfect
[Twitter] timestamp: 2026-02-19 11:55:21.000000000 — Perfect
[Reddit] timestamp: 2026-02-20 09:07:43.000000000 — Perfect
[Reddit] timestamp: 2026-02-20 09:07:43.000000000 — Perfect

 TEST 9: Full RAG Pipeline Simulation 

Query: 'why is spider noir trending right now'

Factual context (for summary):
  [Wikipedia] [Source: Wikipedia | Date: Static Context] Spider-Noir is an upcoming American superhero noir televi...
  [NewsAPI] [Source: NewsAPI | Date: 2026-02-19] Image Courtesy of Marvel Released in 2018, Spider-Man: Into the...
  [NewsAPI] [Source: NewsAPI | Date: 2026-02-18] Image Courtesy of Marvel Comics It seems that the Marvel Cinema...

Social context (for sentiment):
  [Twi

In [155]:
# TEST 10: Final chunk count verification
print("=== TEST 10: Final Chunk Count ===\n")
print(f"Original CSV rows: {len(pd.read_csv('tier1_chunked_final.csv'))}")
print(f"After embedding: {len(df_clean)}")
print(f"After similarity filter (0.30): {len(df_filtered)}")
print(f"Embeddings array shape: {embeddings_filtered.shape}")
print(f"df_filtered and embeddings in sync: {'Good' if len(df_filtered) == embeddings_filtered.shape[0] else 'Check Again'}")

# TEST 11: ChromaDB count matches df_filtered
print("\n=== TEST 11: ChromaDB Sync Check ===\n")
chroma_count = collection.count()
df_count = len(df_filtered)
print(f"df_filtered rows: {df_count}")
print(f"ChromaDB vectors: {chroma_count}")
print(f"In sync: {'Good' if chroma_count == df_count else ' MISMATCH — need to re-store'}")

# TEST 12: Persistence test
print("\n=== TEST 12: ChromaDB Persistence Test ===\n")
# Close and reopen ChromaDB
client2 = chromadb.PersistentClient(path="./chroma_store")
collection2 = client2.get_collection("trend_chunks")
print(f"Reopened ChromaDB count: {collection2.count()}")
print(f"Persistence working: {'Good' if collection2.count() == chroma_count else 'Check Again'}")

# Quick retrieval on reopened collection
results = collection2.query(
    query_embeddings=model.encode(
        ["spider noir trending"], normalize_embeddings=True
    ).tolist(),
    n_results=2
)
print(f"Query on reopened DB returned: {len(results['documents'][0])} results")
print(f"Retrieval working: {'Good ' if len(results['documents'][0]) > 0 else 'Check Again'}")

=== TEST 10: Final Chunk Count ===

Original CSV rows: 253
After embedding: 253
After similarity filter (0.30): 229
Embeddings array shape: (229, 1024)
df_filtered and embeddings in sync: Good

=== TEST 11: ChromaDB Sync Check ===

df_filtered rows: 229
ChromaDB vectors: 229
In sync: Good

=== TEST 12: ChromaDB Persistence Test ===

Reopened ChromaDB count: 229
Persistence working: Good
Query on reopened DB returned: 2 results
Retrieval working: Good 
